In [2]:
import os
import json
import time
import random

import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
MERGED_MODEL_DIR = "outputs/qwen2.5-coder-7b-pr-review-merged"  # fine-tuned model dir

TRAIN_PATH = "merged/train.jsonl"
VAL_PATH = "merged/val.jsonl"

N_TRAIN_USED = 200
N_VAL_USED = 50

N_EVAL_SAMPLES = 30      
MAX_NEW_TOKENS = 300       
RANDOM_SEED = 42

OUTPUT_DIR = "evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cpu")
print("Running evaluation on CPU")

random.seed(RANDOM_SEED)


Running evaluation on CPU


In [3]:
print("Loading base model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float32,
    trust_remote_code=True,
)
base_model.eval()
base_model.to(device)

print("Loading fine-tuned model...")
ft_tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
ft_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    torch_dtype=torch.float32,
)
ft_model.eval()
ft_model.to(device)

print("Both models loaded.")


Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading fine-tuned model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both models loaded.


In [4]:
raw_val = load_dataset("json", data_files={"validation": VAL_PATH})["validation"]
print(f"Total validation examples available: {len(raw_val)}")

held_out = raw_val.select(range(N_VAL_USED, len(raw_val)))
print(f"Held-out (unseen) examples: {len(held_out)}")

sample_indices = random.sample(range(len(held_out)), min(N_EVAL_SAMPLES, len(held_out)))
eval_examples = held_out.select(sample_indices)
print(f"Evaluating on {len(eval_examples)} held-out PRs")


Total validation examples available: 27402
Held-out (unseen) examples: 27352
Evaluating on 30 held-out PRs


In [5]:
CATEGORY_TEST_SET = [
    {
        "category": "security",
        "input": (
            "Repository: example-org/example-service\n"
            "Language: Python\n"
            "PR Title: Add user password reset endpoint\n"
            "PR Description: Adds a new POST /reset-password endpoint.\n"
            "File: app/routes/auth.py\n\n"
            "Diff:\n"
            "+@app.route(\"/reset-password\", methods=[\"POST\"])\n"
            "+def reset_password():\n"
            "+    email = request.json[\"email\"]\n"
            "+    user = db.query(f\"SELECT * FROM users WHERE email = '{email}'\")\n"
            "+    token = str(random.randint(100000, 999999))\n"
            "+    cache.set(email, token)\n"
        ),
        "reference": (
            "This code is vulnerable to SQL injection because the email is "
            "interpolated directly into the query string; use parameterized "
            "queries instead. Also, using random.randint for a reset token "
            "is insecure -- use a cryptographically secure random generator "
            "such as secrets.token_urlsafe."
        ),
    },
    {
        "category": "performance",
        "input": (
            "Repository: example-org/example-service\n"
            "Language: Python\n"
            "PR Title: Fetch user orders for dashboard\n"
            "PR Description: Loads all orders per user for the dashboard view.\n"
            "File: app/dashboard.py\n\n"
            "Diff:\n"
            "+def get_dashboard_data(users):\n"
            "+    result = []\n"
            "+    for user in users:\n"
            "+        orders = db.query(f\"SELECT * FROM orders WHERE user_id = {user.id}\")\n"
            "+        result.append({\"user\": user, \"orders\": orders})\n"
            "+    return result\n"
        ),
        "reference": (
            "This introduces an N+1 query problem -- one query is issued per "
            "user in a loop. Consider a single batched query using "
            "WHERE user_id IN (...) or a join to fetch all orders at once."
        ),
    },
    {
        "category": "code_quality",
        "input": (
            "Repository: example-org/example-service\n"
            "Language: Python\n"
            "PR Title: Add discount calculation\n"
            "PR Description: Adds discount logic for checkout.\n"
            "File: app/checkout.py\n\n"
            "Diff:\n"
            "+def calc(a,b,c,d):\n"
            "+    if c==1:\n"
            "+        return a-b\n"
            "+    elif c==2:\n"
            "+        return a-(a*b/100)\n"
            "+    else:\n"
            "+        return a\n"
        ),
        "reference": (
            "Function and parameter names are unclear (calc, a, b, c, d) -- "
            "rename to something descriptive like calculate_discount(price, "
            "discount_value, discount_type). The unused parameter d and "
            "magic numbers (1, 2) for discount type should be replaced with "
            "named constants or an enum."
        ),
    },
    {
        "category": "architecture",
        "input": (
            "Repository: example-org/example-service\n"
            "Language: Python\n"
            "PR Title: Add email sending to order controller\n"
            "PR Description: Sends confirmation email directly from the controller.\n"
            "File: app/controllers/order_controller.py\n\n"
            "Diff:\n"
            "+import smtplib\n"
            "+def create_order(request):\n"
            "+    order = Order.create(request.data)\n"
            "+    smtp = smtplib.SMTP(\"smtp.example.com\")\n"
            "+    smtp.sendmail(\"noreply@example.com\", order.user.email, \"Order confirmed\")\n"
            "+    return order\n"
        ),
        "reference": (
            "Sending email directly from the controller couples HTTP request "
            "handling with infrastructure concerns and makes this hard to "
            "test or reuse. Move email sending into a dedicated service or "
            "notification layer, and consider making it async so order "
            "creation isn't blocked on SMTP latency."
        ),
    },
    {
        "category": "best_practices",
        "input": (
            "Repository: example-org/example-service\n"
            "Language: Python\n"
            "PR Title: Add file upload handling\n"
            "PR Description: Handles user file upload.\n"
            "File: app/upload.py\n\n"
            "Diff:\n"
            "+def handle_upload(file):\n"
            "+    try:\n"
            "+        data = file.read()\n"
            "+        save(data)\n"
            "+    except:\n"
            "+        pass\n"
        ),
        "reference": (
            "Using a bare except that silently swallows all exceptions is a "
            "bad practice -- it hides real errors (disk full, permission "
            "denied, corrupt file, etc.) and makes debugging difficult. "
            "Catch specific exceptions and log or handle them appropriately."
        ),
    },
]

print(f"Category test set size: {len(CATEGORY_TEST_SET)}")


Category test set size: 5


In [6]:
SYSTEM_PROMPT = (
    "You are an expert software engineer performing thorough, constructive "
    "code reviews on GitHub Pull Requests."
)

DEFAULT_INSTRUCTION = (
    "You are an experienced software engineer performing a code review. "
    "Given the following pull request context and code diff, write a "
    "concise, helpful review comment."
)


def generate_review(model, tokenizer, pr_input, instruction=DEFAULT_INSTRUCTION,
                     max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\n{pr_input}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


In [ ]:
from tqdm.auto import tqdm

heldout_results = []

for i, example in enumerate(tqdm(eval_examples, desc="Held-out PRs")):
    start = time.time()

    base_output = generate_review(
        base_model, base_tokenizer, example["input"], example["instruction"]
    )
    ft_output = generate_review(
        ft_model, ft_tokenizer, example["input"], example["instruction"]
    )

    heldout_results.append({
        "index": i,
        "repo": example["metadata"].get("repo"),
        "language": example["metadata"].get("language"),
        "reference": example["output"],
        "base_output": base_output,
        "finetuned_output": ft_output,
    })

    elapsed = time.time() - start
    print(f"[{i+1}/{len(eval_examples)}] done in {elapsed:.1f}s")

heldout_df = pd.DataFrame(heldout_results)
heldout_df.to_json(os.path.join(OUTPUT_DIR, "heldout_generations.jsonl"),
                    orient="records", lines=True, force_ascii=False)
print(f"\nSaved {len(heldout_df)} held-out generations.")


Held-out PRs:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
category_results = []

for item in tqdm(CATEGORY_TEST_SET, desc="Category test set"):
    base_output = generate_review(base_model, base_tokenizer, item["input"])
    ft_output = generate_review(ft_model, ft_tokenizer, item["input"])

    category_results.append({
        "category": item["category"],
        "reference": item["reference"],
        "base_output": base_output,
        "finetuned_output": ft_output,
    })

category_df = pd.DataFrame(category_results)
category_df.to_json(os.path.join(OUTPUT_DIR, "category_generations.jsonl"),
                     orient="records", lines=True, force_ascii=False)
print(f"Saved {len(category_df)} category generations.")


In [ ]:
import re
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bert_score

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def normalize(text):
    return re.sub(r"\s+", " ", (text or "").strip().lower())


def exact_match(preds, refs):
    matches = sum(normalize(p) == normalize(r) for p, r in zip(preds, refs))
    return matches / len(preds) if preds else 0.0


def rouge_l(preds, refs):
    scores = [rouge.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
    return sum(scores) / len(scores) if scores else 0.0


def bleu(preds, refs):
    if not preds:
        return 0.0
    bleu_result = sacrebleu.corpus_bleu(preds, [refs])
    return bleu_result.score


def bertscore_f1(preds, refs):
    if not preds:
        return 0.0
    P, R, F1 = bert_score(preds, refs, lang="en", verbose=False)
    return F1.mean().item()


def compute_all_metrics(preds, refs):
    return {
        "exact_match": exact_match(preds, refs),
        "rougeL": rouge_l(preds, refs),
        "bleu": bleu(preds, refs),
        "bertscore_f1": bertscore_f1(preds, refs),
    }


In [ ]:
heldout_refs = heldout_df["reference"].tolist()
heldout_base_preds = heldout_df["base_output"].tolist()
heldout_ft_preds = heldout_df["finetuned_output"].tolist()

print("Computing metrics for held-out set")
heldout_base_metrics = compute_all_metrics(heldout_base_preds, heldout_refs)
heldout_ft_metrics = compute_all_metrics(heldout_ft_preds, heldout_refs)

heldout_summary = pd.DataFrame([
    {"model": "base", **heldout_base_metrics},
    {"model": "fine-tuned", **heldout_ft_metrics},
])
print("\n=== Held-out validation set: Base vs Fine-tuned ===")
print(heldout_summary.to_string(index=False))


In [ ]:
category_refs = category_df["reference"].tolist()
category_base_preds = category_df["base_output"].tolist()
category_ft_preds = category_df["finetuned_output"].tolist()

print("Computing metrics for category test set...")
category_base_metrics = compute_all_metrics(category_base_preds, category_refs)
category_ft_metrics = compute_all_metrics(category_ft_preds, category_refs)

category_summary = pd.DataFrame([
    {"model": "base", **category_base_metrics},
    {"model": "fine-tuned", **category_ft_metrics},
])
print("\n=== Category test set: Base vs Fine-tuned ===")
print(category_summary.to_string(index=False))
